# Phase 4: unlearning English from θ_b (Gate 4)

**Only after Gate 3 is approved.** Upload the latest `colab/code_bundle.tar.gz` first.
Runs the 6-run grid of `configs/unlearn_v2.yaml` (deviations D5, D6): M1 control, M2 gradient ascent and
4 × M3 NegGrad+ (lr × λ). Each run takes at most 1000 steps. Evaluation and checkpoint every 5 steps for the first
100 steps, then every 100 steps. M1 is reused from the first grid.
The unlearning code never loads θ_ref (`tests/test_isolation.py`). The operating points are chosen afterwards
by `select_points.py`, which is evaluation-side code.

In [ ]:
# 1. Settings + GPU check
SEED = 0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive. MyDrive/unlearn/{checkpoints,results} is the working copy from Phase 3 on.
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/unlearn'
import os
for d in ('checkpoints', 'results', 'gate_packs'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print(sorted(os.listdir(DRIVE)))

In [ ]:
# 3. Data bundle (unchanged since Phase 0) + latest code bundle on top; check the data SHAs;
#    link checkpoints/ and results/ to Drive so every script writes there directly.
!rm -rf /content/unlearn && mkdir -p /content/unlearn
!tar xzf {DRIVE}/unlearn_bundle.tar.gz -C /content/unlearn
!tar xzf {DRIVE}/code_bundle.tar.gz -C /content/unlearn
%cd /content/unlearn
!cd data/processed && sha256sum -c --quiet MANIFEST.sha256 && echo "data OK"
!rm -rf checkpoints results && ln -s {DRIVE}/checkpoints checkpoints && ln -s {DRIVE}/results results

In [ ]:
# 4. Dependencies + tests. tests/test_data.py is not run here: it needs the *_titles.txt files, which are
#    not in the bundle. It passed on the Mac at Gate 0, and cell 3 already proved the .bin files are
#    byte-identical to the Mac copy (SHA-256). Smoke-data tests are skipped (no smoke data on Colab).
!pip -q install pyyaml
!python -m pytest -q --ignore=tests/test_data.py 2>&1 | tail -2

In [ ]:
# 5. Bring the Mac's results into Drive (never overwrites), then re-hash every protected checkpoint.
#    Drive cannot chmod, so this SHA check is what protects θ_ref. It must print OK, never raise.
!python scripts/colab_sync.py merge --src results_mac --dst results
!python scripts/colab_sync.py verify

In [ ]:
# 6. Unlearning grid v2 (deviation D6: eval + checkpoint every 5 steps for the first 100 steps).
#    M1 is reused from v1 (skipped); M2 and M3 rerun with tag v2. Finished runs are skipped.
!python scripts/unlearn.py --grid --config configs/unlearn_v2.yaml --seed {SEED} 2>&1 | grep -v Warning

In [ ]:
# 7. Operating points + trajectory figure. Paste this cell's printed output into the chat for Gate 4.
!python scripts/select_points.py --seed {SEED} --config configs/unlearn_v2.yaml
!python scripts/plots.py trajectory --seed {SEED}
from IPython.display import Image; Image(f'results/figures/trajectory_s{SEED}.png')

In [ ]:
# 8. Samples at the practitioner point of every run (validation prompts, no test data yet)
import json, os
pts = json.load(open(f'results/operating_points_s{SEED}.json'))
done_ckpts = {f'checkpoints/theta_b_s{SEED}.pt': f'theta_b_s{SEED}'}   # θ_b was evaluated at Gate 3
for run, info in pts['runs'].items():
    ck = info['practitioner']['ckpt']
    if ck in done_ckpts:
        print(f'==== {run}: practitioner point = {done_ckpts[ck]} (step {info["practitioner"]["step"]}), not evaluated again')
        continue
    name = f'{run}@prac_val'
    done_ckpts[ck] = name
    if not os.path.exists(f'results/samples/{name}.json'):
        ck = info['practitioner']['ckpt']
        !python scripts/eval_checkpoint.py --ckpt {ck} --name {name} --splits pt_val en_val --prompts val 2>&1 | grep -v Warning
    s = json.load(open(f'results/samples/{name}.json'))
    print(f'==== {run} (step {info["practitioner"]["step"]})')
    for lang in ('pt', 'en'):
        print(f'  [{lang}] ...{s[lang][0]["prompt"][-30:]!r} -> {s[lang][0]["generation"][:160]!r}')

In [ ]:
# Gate pack: re-check the hashes, then zip results/ (small files only, no checkpoints).
# Download MyDrive/unlearn/gate_packs/gate4_results.zip and unzip it into the project as colab_results/gate4/
!python scripts/colab_sync.py verify
!cd {DRIVE} && rm -f gate_packs/gate4_results.zip && zip -qr gate_packs/gate4_results.zip results && ls -la gate_packs